# Guia de testes — Pesquisa Informada

Este notebook serve para testar a componente de **Pesquisa Informada** do projeto: `model.py` e `pesquisa.py`.

O objetivo é confirmar que os módulos carregam os dados, que os algoritmos encontram rotas viáveis e que é possível comparar **UCS, Greedy com H1/H2 e A* com H1/H2** através da distância, dos nós expandidos e do tempo de execução.

> **Âmbito:** estes testes avaliam uma rota de **um veículo num dia**, com um conjunto de encomendas definido. Não substituem o CSP do Tiago nem a integração dos dois veículos e dos dias 30 e 31, que serão tratados nas restantes componentes do projeto.

Executa as células pela ordem apresentada. As células de configuração e de seleção de dados podem ser alteradas para testar outro conjunto de dados, veículo ou número de encomendas.

## 1. Preparar o ambiente e localizar o projeto

### IMPORTANTE: A estrutura esperada é semelhante a esta (retirar temporariamente da pasta e meter na raiz com model.py antes de testar): 

```text
IA26_P01_G02/
│
├── model.py
├── pesquisa.py
├── Guia_Testes_Pesquisa_Informada.ipynb
│
├── docs/
│   ├── data_tiny/
│   │   ├── ... (12 ficheiros CSV)
│   │
│   └── data_small/
│       ├── ... (12 ficheiros CSV)
│
└── resultados/
```

O exemplo começa pelo conjunto `tiny`, porque é mais pequeno e adequado aos primeiros testes. Confirma que os CSV oficiais estão na pasta indicada.

In [84]:
from pathlib import Path
import sys
import subprocess
import inspect
import importlib
import pandas as pd
from IPython.display import display

def encontrar_raiz_projeto(pasta_inicial: Path) -> Path:
    """Procura a pasta que contém model.py e pesquisa.py."""
    for pasta in (pasta_inicial, *pasta_inicial.parents):
        if (pasta / "model.py").is_file() and (pasta / "pesquisa.py").is_file():
            return pasta
    raise FileNotFoundError(
        "Não encontrei uma pasta com model.py e pesquisa.py. "
        "Coloca o notebook dentro da pasta do projeto ou numa subpasta."
    )

RAIZ = encontrar_raiz_projeto(Path.cwd())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

# Altera este caminho caso os teus CSV estejam noutra pasta.
PASTA_DADOS = RAIZ / "docs" / "data_tiny"

# Parâmetros do teste inicial.
ID_VEICULO = 1
MAX_ENCOMENDAS = 2
MAX_NOS_EXPANDIDOS = 50_000

print("Pasta do projeto:", RAIZ)
print("Pasta dos dados:", PASTA_DADOS)
print("Notebook preparado.")

Pasta do projeto: c:\Users\fabix\Documents\MyCodeStuff\IA26_P01_G02
Pasta dos dados: c:\Users\fabix\Documents\MyCodeStuff\IA26_P01_G02\docs\data_tiny
Notebook preparado.


## 2. Teste 1 — Verificar a sintaxe e as importações

Primeiro, o Python compila os dois ficheiros. Se o comando terminar sem erros, não foram encontrados erros de sintaxe.

Depois, importa os módulos. Uma importação bem-sucedida confirma também que as dependências necessárias para carregar os módulos estão disponíveis neste ambiente.

In [85]:
ficheiros_python = [RAIZ / "model.py", RAIZ / "pesquisa.py"]

for ficheiro in ficheiros_python:
    if not ficheiro.is_file():
        raise FileNotFoundError(f"Ficheiro não encontrado: {ficheiro}")

compilacao = subprocess.run(
    [sys.executable, "-m", "py_compile", *map(str, ficheiros_python)],
    cwd=RAIZ,
    capture_output=True,
    text=True
)

if compilacao.returncode == 0:
    print("Compilação OK — model.py e pesquisa.py não apresentam erros de sintaxe.")
else:
    print(compilacao.stdout)
    print(compilacao.stderr)
    raise RuntimeError("A compilação dos módulos falhou.")

import model
import pesquisa

# Recarrega os módulos caso tenham sido alterados depois de uma execução anterior.
importlib.reload(model)
importlib.reload(pesquisa)

print("Importação OK — model e pesquisa foram carregados.")

Compilação OK — model.py e pesquisa.py não apresentam erros de sintaxe.
Importação OK — model e pesquisa foram carregados.


### Consultar as funções disponíveis

Esta célula mostra as assinaturas das funções principais. É útil para confirmar os argumentos disponíveis na versão dos ficheiros que estás a testar.

In [86]:
funcoes_a_consultar = [
    (model, "Problem"),
    (pesquisa, "selecionar_itens_veiculo_dia"),
    (pesquisa, "comparar_algoritmos"),
    (pesquisa, "executar_pesquisa"),
    (pesquisa, "ucs"),
    (pesquisa, "greedy_search"),
    (pesquisa, "a_star_search"),
    (pesquisa, "h1_pedido_mais_caro"),
    (pesquisa, "h2_mst"),
]

for modulo, nome in funcoes_a_consultar:
    objeto = getattr(modulo, nome, None)
    if objeto is None:
        print(f"{modulo.__name__}.{nome}: não encontrada nesta versão")
    else:
        try:
            print(f"{modulo.__name__}.{nome}{inspect.signature(objeto)}")
        except (TypeError, ValueError):
            print(f"{modulo.__name__}.{nome}: disponível")

model.Problem(pasta: str)
pesquisa.selecionar_itens_veiculo_dia(problem: 'Problem', vehicle_id: 'int', dia: 'date', max_encomendas: 'Optional[int]' = None) -> 'Set[int]'
pesquisa.comparar_algoritmos(problem: 'Problem', vehicle_id: 'int', itens_objetivo: 'Set[int]', hora_partida: 'datetime', itens_ja_no_armazem: 'Optional[Dict[int, datetime]]' = None, max_nos_expandidos: 'Optional[int]' = None) -> 'Tuple[pd.DataFrame, Dict[str, Dict[str, object]]]'
pesquisa.executar_pesquisa(problem: 'Problem', vehicle_id: 'int', itens_objetivo: 'Set[int]', hora_partida: 'datetime', algoritmo: 'str' = 'a_star', heuristica: 'Optional[Heuristica]' = None, itens_ja_no_armazem: 'Optional[Dict[int, datetime]]' = None, max_nos_expandidos: 'Optional[int]' = None) -> 'Dict[str, object]'
pesquisa.ucs(problem: 'Problem', vehicle_id: 'int', itens_objetivo: 'Set[int]', hora_partida: 'datetime', itens_ja_no_armazem: 'Optional[Dict[int, datetime]]' = None, max_nos_expandidos: 'Optional[int]' = None) -> 'Dict[str, obj

## 3. Teste 2 — Carregar e verificar os CSV

A pasta de dados deve conter os **12 ficheiros CSV** esperados pelo modelo. A célula seguinte lista os ficheiros encontrados e avisa se o número for diferente de 12.

Depois, cria o objeto `Problem`, que carrega os dados e prepara a representação do problema.

In [87]:
if not PASTA_DADOS.is_dir():
    raise FileNotFoundError(
        f"A pasta de dados não existe: {PASTA_DADOS}\n"
        "Corrige PASTA_DADOS na secção 1."
    )

ficheiros_csv = sorted(PASTA_DADOS.glob("*.csv"))

display(pd.DataFrame({
    "Ficheiro CSV": [f.name for f in ficheiros_csv]
}))

print(f"Total de CSV encontrados: {len(ficheiros_csv)}")

if len(ficheiros_csv) != 12:
    print(
        "ATENÇÃO: eram esperados 12 CSV. Confirma se a pasta está completa "
        "e se os ficheiros estão no nível correto."
    )
else:
    print("Número de CSV esperado encontrado.")

,Ficheiro CSV
0,collection_centers.csv
1,customers.csv
2,delivery_locations.csv
3,distance_matrix.csv
4,localities.csv
5,locations.csv
6,order_items.csv
7,orders.csv
8,producers.csv
9,products.csv


Total de CSV encontrados: 12
Número de CSV esperado encontrado.


In [88]:
# Carregar os dados do conjunto selecionado.
problem = model.Problem(str(PASTA_DADOS))

print("Dados carregados com sucesso.")
print("Número de pedidos/itens registados:", len(problem.pedidos))

# Informação adicional disponível no objeto Problem.
for atributo in ("veiculos", "paragens", "distancias", "tempos"):
    if hasattr(problem, atributo):
        valor = getattr(problem, atributo)
        try:
            quantidade = len(valor)
        except TypeError:
            quantidade = "n/a"
        print(f"{atributo}: disponível ({quantidade})")

Dados carregados com sucesso.
Número de pedidos/itens registados: 40
veiculos: disponível (2)
paragens: disponível (9)


### Testes manuais dos scripts (opcional)

Também podes executar os ficheiros diretamente no terminal, a partir da pasta principal:

```bash
python model.py
python pesquisa.py
```

O comportamento depende do bloco de demonstração existente no fim de cada ficheiro. A célula seguinte permite fazer essa verificação a partir do notebook. Está desativada por defeito para evitar executar demonstrações que possam demorar mais tempo ou repetir trabalho.

Muda `EXECUTAR_SCRIPTS = True` apenas quando quiseres executar esses testes.

In [89]:
EXECUTAR_SCRIPTS = False  # Muda para True para executar model.py e pesquisa.py.

if EXECUTAR_SCRIPTS:
    for nome_script in ("model.py", "pesquisa.py"):
        print(f"\n===== A executar {nome_script} =====")
        processo = subprocess.run(
            [sys.executable, str(RAIZ / nome_script)],
            cwd=RAIZ,
            capture_output=True,
            text=True,
            timeout=180
        )
        print(processo.stdout)
        if processo.stderr:
            print("STDERR:")
            print(processo.stderr)
        print("Código de saída:", processo.returncode)
else:
    print("Execução direta dos scripts ignorada. Para a ativar, define EXECUTAR_SCRIPTS = True.")

Execução direta dos scripts ignorada. Para a ativar, define EXECUTAR_SCRIPTS = True.


## 4. Teste 3 — Escolher encomendas para uma rota

Para comparar os algoritmos de forma justa, todos têm de receber exatamente:

- o mesmo veículo;
- o mesmo dia;
- os mesmos itens/encomendas;
- a mesma hora de partida.

Começamos por selecionar até duas encomendas completas do primeiro dia disponível no conjunto `tiny`. Isto mantém o teste relativamente pequeno.

A função `selecionar_itens_veiculo_dia()` devolve os IDs dos itens elegíveis. O modelo considera as encomendas completas, em vez de separar arbitrariamente os seus itens.

In [90]:
# Identificar os dias disponíveis nos pedidos.
pedidos = list(problem.pedidos.values())

if not pedidos:
    raise RuntimeError("O conjunto de dados não contém pedidos.")

if not hasattr(pedidos[0], "dia_recolha"):
    raise AttributeError(
        "Não encontrei o atributo 'dia_recolha' nos objetos de pedido. "
        "Consulta a célula das assinaturas/atributos e ajusta a seleção ao modelo atual."
    )

DIA_TESTE = min(pedido.dia_recolha for pedido in pedidos)

itens_objetivo = pesquisa.selecionar_itens_veiculo_dia(
    problem,
    vehicle_id=ID_VEICULO,
    dia=DIA_TESTE,
    max_encomendas=MAX_ENCOMENDAS
)

print("Veículo:", ID_VEICULO)
print("Dia selecionado:", DIA_TESTE)
print("Itens selecionados:", sorted(itens_objetivo))
print("Total de itens selecionados:", len(itens_objetivo))

if not itens_objetivo:
    print(
        "Não foram encontradas encomendas elegíveis. "
        "Experimenta outro veículo, outro dia ou confirma os dados CSV."
    )

Veículo: 1
Dia selecionado: 2026-10-30
Itens selecionados: [11, 12, 13, 30, 31]
Total de itens selecionados: 5


In [91]:
if itens_objetivo:
    hora_partida = min(
        problem.pedidos[item_id].availability_start
        for item_id in itens_objetivo
    )
    print("Hora de partida utilizada:", hora_partida)
else:
    hora_partida = None
    print("Sem hora de partida: não há itens para pesquisar.")

Hora de partida utilizada: 2026-10-30 07:15:00


## 5. Teste 4 — Comparar UCS, Greedy e A*

Esta é a comparação principal da componente de Pesquisa Informada.

São executadas cinco configurações:

1. UCS — pesquisa de custo uniforme, sem heurística;
2. Greedy H1 — utiliza a heurística do pedido mais caro;
3. Greedy H2 — utiliza a heurística baseada na árvore geradora mínima (MST);
4. A* H1 — combina o custo acumulado com H1;
5. A* H2 — combina o custo acumulado com H2.

O limite de nós é uma proteção contra pesquisas demasiado demoradas. Para um teste de optimalidade no conjunto `tiny`, podes aumentar o limite ou usar `None`, desde que a dimensão do problema o permita.

In [92]:
if not itens_objetivo:
    print("A comparação não foi executada porque não existem itens selecionados.")
    tabela = None
    detalhes = None
else:
    tabela, detalhes = pesquisa.comparar_algoritmos(
        problem,
        vehicle_id=ID_VEICULO,
        itens_objetivo=itens_objetivo,
        hora_partida=hora_partida,
        max_nos_expandidos=MAX_NOS_EXPANDIDOS
    )

    print("Comparação concluída.")
    display(tabela)

Comparação concluída.


,Algoritmo,Sucesso,Viável (verificador),Distância (km),Nós expandidos,Nós gerados,Tempo (s),Limite atingido,Rota
0,UCS,True,True,179.47,488,801,0.026564,False,4 -> 3 -> 2 -> 1 -> 4 -> 8 -> 7 -> 4
1,Greedy — H1 pedido mais caro,True,True,234.43,47,118,0.004721,False,4 -> 2 -> 1 -> 3 -> 2 -> 4 -> 7 -> 8 -> 4
2,Greedy — H2 MST,True,True,199.14,20,71,0.002504,False,4 -> 3 -> 2 -> 1 -> 4 -> 7 -> 4 -> 8 -> 4
3,A* — H1 pedido mais caro,True,True,179.47,51,155,0.005521,False,4 -> 3 -> 2 -> 1 -> 4 -> 8 -> 7 -> 4
4,A* — H2 MST,True,True,179.47,102,266,0.013647,False,4 -> 3 -> 2 -> 1 -> 4 -> 8 -> 7 -> 4


## 6. Teste 5 — Interpretar e validar os resultados

A tabela deverá apresentar, pelo menos, o algoritmo, o sucesso, a viabilidade, a distância, os nós expandidos, os nós gerados, o tempo e a indicação de limite atingido.

### O que verificar

- `Sucesso = True`: o algoritmo encontrou uma solução.
- `Viável (verificador) = True`: o verificador aceitou a rota.
- `Limite atingido = False`: a pesquisa não parou por atingir o limite configurado.
- A rota começa e termina no armazém.
- Todos os itens atribuídos ao teste foram entregues.
- Não existem violações de capacidade, janelas horárias ou sequência logística.

Se `Limite atingido = True`, a ausência de solução não prova que o problema é impossível: a pesquisa pode simplesmente ter sido interrompida.

**Nota sobre optimalidade:** o facto de A* encontrar a mesma distância que UCS é um bom resultado, mas não prova, sozinho, que A* garante sempre o ótimo. Para essa garantia é necessário confirmar a admissibilidade da heurística e o tratamento correto dos estados repetidos. O UCS também só serve como referência ótima se terminar a pesquisa sem cortes e com o modelo correto.

In [93]:
if tabela is not None:
    display(tabela)

    colunas_essenciais = [
        "Algoritmo",
        "Sucesso",
        "Viável (verificador)",
        "Distância (km)",
        "Nós expandidos",
        "Tempo (s)",
        "Limite atingido",
    ]

    faltam = [coluna for coluna in colunas_essenciais if coluna not in tabela.columns]
    if faltam:
        print("ATENÇÃO — colunas não encontradas:", faltam)
        print("Colunas disponíveis:", list(tabela.columns))
    else:
        print("\nResumo das verificações:")
        print("Todos encontraram solução:",
              bool(tabela["Sucesso"].fillna(False).all()))
        print("Todas as rotas são viáveis:",
              bool(tabela["Viável (verificador)"].fillna(False).all()))
        print("Nenhum algoritmo atingiu o limite:",
              bool((~tabela["Limite atingido"].fillna(True)).all()))

        if tabela["Nós expandidos"].notna().all():
            linha_ucs = tabela[tabela["Algoritmo"].astype(str).str.upper().eq("UCS")]
            linhas_astar = tabela[tabela["Algoritmo"].astype(str).str.startswith("A*")]

            if not linha_ucs.empty and not linhas_astar.empty:
                nos_ucs = float(linha_ucs.iloc[0]["Nós expandidos"])
                for _, linha in linhas_astar.iterrows():
                    nos_astar = float(linha["Nós expandidos"])
                    if nos_ucs > 0:
                        reducao = (nos_ucs - nos_astar) / nos_ucs * 100
                        print(
                            f"{linha['Algoritmo']}: redução de "
                            f"{reducao:.1f}% nos nós expandidos face ao UCS."
                        )
else:
    print("Ainda não existem resultados para validar.")

,Algoritmo,Sucesso,Viável (verificador),Distância (km),Nós expandidos,Nós gerados,Tempo (s),Limite atingido,Rota
0,UCS,True,True,179.47,488,801,0.026564,False,4 -> 3 -> 2 -> 1 -> 4 -> 8 -> 7 -> 4
1,Greedy — H1 pedido mais caro,True,True,234.43,47,118,0.004721,False,4 -> 2 -> 1 -> 3 -> 2 -> 4 -> 7 -> 8 -> 4
2,Greedy — H2 MST,True,True,199.14,20,71,0.002504,False,4 -> 3 -> 2 -> 1 -> 4 -> 7 -> 4 -> 8 -> 4
3,A* — H1 pedido mais caro,True,True,179.47,51,155,0.005521,False,4 -> 3 -> 2 -> 1 -> 4 -> 8 -> 7 -> 4
4,A* — H2 MST,True,True,179.47,102,266,0.013647,False,4 -> 3 -> 2 -> 1 -> 4 -> 8 -> 7 -> 4



Resumo das verificações:
Todos encontraram solução: True
Todas as rotas são viáveis: True
Nenhum algoritmo atingiu o limite: True
A* — H1 pedido mais caro: redução de 89.5% nos nós expandidos face ao UCS.
A* — H2 MST: redução de 79.1% nos nós expandidos face ao UCS.


### Como interpretar os comportamentos

- **UCS** pode expandir mais nós porque ordena a pesquisa apenas pelo custo já percorrido (`g`).
- **Greedy** usa apenas a estimativa heurística (`h`). Pode ser rápido, mas não garante a rota de menor distância.
- **A\*** usa `g + h`. Uma boa heurística pode orientar a pesquisa e reduzir os nós expandidos, mantendo o custo ótimo quando as condições teóricas estão cumpridas.
- H1 pode funcionar melhor num cenário e H2 noutro. Não se deve concluir que uma heurística é sempre superior com base numa única execução.

Os tempos apresentados podem ser muito pequenos. Para conclusões sobre desempenho, repete os testes e compara também os tempos médios ou medianos.

## 7. Consultar as rotas e as ações

A tabela resume os resultados, mas para a integração com o Alexandre e o André também é importante guardar a sequência de visitas e as ações logísticas.

- `pickup`: recolha de produtos num produtor;
- `warehouse`: passagem pelo armazém;
- `load`: carregamento de encomendas;
- `delivery`: entrega no destino.

A célula seguinte mostra as rotas e as ações devolvidas por cada algoritmo.

In [94]:
if detalhes is not None:
    def nome_paragem_seguro(problem, paragem_id):
        """Obtém o nome da paragem; se não existir, apresenta o respetivo ID."""
        paragens = getattr(problem, "paragens", {})
        dados = paragens.get(paragem_id) if hasattr(paragens, "get") else None

        if isinstance(dados, dict):
            return (
                dados.get("nome")
                or dados.get("name")
                or dados.get("location_name")
                or str(paragem_id)
            )

        if dados is not None:
            return (
                getattr(dados, "nome", None)
                or getattr(dados, "name", None)
                or str(paragem_id)
            )

        return str(paragem_id)

    for nome_algoritmo, resultado in detalhes.items():
        print(f"\n===== {nome_algoritmo} =====")

        rota = resultado.get("rota_paragens", [])
        acoes = resultado.get("acoes", [])

        if rota:
            rota_nomes = [nome_paragem_seguro(problem, p) for p in rota]
            print("Rota:", " -> ".join(rota_nomes))
        else:
            print("Rota não disponível neste resultado.")

        print("Ações:", acoes)

        if "sucesso" in resultado:
            print("Sucesso:", resultado["sucesso"])
        if "viavel" in resultado:
            print("Viável:", resultado["viavel"])
else:
    print("Não existem detalhes de rotas para apresentar.")


===== UCS =====
Rota: Centro de Recolha 1 -> Horta Morais -> Campos Leite -> Hortas Valente -> Centro de Recolha 1 -> Casa 4 -> Casa 3 -> Centro de Recolha 1
Ações: [('pickup', 11), ('pickup', 12), ('pickup', 30), ('pickup', 13), ('pickup', 31), ('warehouse', 4), ('load', 11), ('load', 12), ('load', 13), ('load', 30), ('load', 31), ('delivery', 30), ('delivery', 31), ('delivery', 11), ('delivery', 12), ('delivery', 13), ('warehouse', 4)]
Sucesso: True

===== Greedy — H1 pedido mais caro =====
Rota: Centro de Recolha 1 -> Campos Leite -> Hortas Valente -> Horta Morais -> Campos Leite -> Centro de Recolha 1 -> Casa 3 -> Casa 4 -> Centro de Recolha 1
Ações: [('pickup', 30), ('pickup', 13), ('pickup', 31), ('pickup', 11), ('pickup', 12), ('warehouse', 4), ('load', 11), ('load', 12), ('load', 13), ('load', 30), ('load', 31), ('delivery', 11), ('delivery', 12), ('delivery', 13), ('delivery', 30), ('delivery', 31), ('warehouse', 4)]
Sucesso: True

===== Greedy — H2 MST =====
Rota: Centro de 

## 8. Guardar os resultados em ficheiros

Os resultados são devolvidos em memória pelo Python. Para os poderes usar no relatório ou partilhar, guarda:

1. a tabela comparativa num CSV;
2. as rotas e as ações num segundo CSV.

Os ficheiros são criados na pasta `pesquisa_informada/resultados_pesq_infor/` do projeto e podem ser abertos no Excel.

In [95]:
if tabela is not None:
    pasta_resultados = RAIZ / "pesquisa_informada" / "resultados_pesq_infor"
    pasta_resultados.mkdir(parents=True, exist_ok=True)

    ficheiro_tabela = pasta_resultados / f"comparacao_veiculo{ID_VEICULO}.csv"
    tabela.to_csv(ficheiro_tabela, index=False, encoding="utf-8-sig")
    print("Tabela guardada em:", ficheiro_tabela)

    linhas_rotas = []
    if detalhes is not None:
        for nome_algoritmo, resultado in detalhes.items():
            rota = resultado.get("rota_paragens", [])
            acoes = resultado.get("acoes", [])

            linhas_rotas.append({
                "Algoritmo": nome_algoritmo,
                "Veículo": ID_VEICULO,
                "Dia": str(DIA_TESTE),
                "Rota_IDs": " -> ".join(map(str, rota)),
                "Ações": repr(acoes),
                "Sucesso": resultado.get("sucesso"),
                "Viável": resultado.get("viavel"),
            })

    if linhas_rotas:
        df_rotas = pd.DataFrame(linhas_rotas)
        ficheiro_rotas = pasta_resultados / f"rotas_veiculo{ID_VEICULO}.csv"
        df_rotas.to_csv(ficheiro_rotas, index=False, encoding="utf-8-sig")
        print("Rotas e ações guardadas em:", ficheiro_rotas)
else:
    print("Não há resultados para guardar.")

Tabela guardada em: c:\Users\fabix\Documents\MyCodeStuff\IA26_P01_G02\pesquisa_informada\resultados_pesq_infor\comparacao_veiculo1.csv
Rotas e ações guardadas em: c:\Users\fabix\Documents\MyCodeStuff\IA26_P01_G02\pesquisa_informada\resultados_pesq_infor\rotas_veiculo1.csv


## 9. Repetir a experiência com outros dados

Depois de concluíres o teste inicial, podes alterar na secção 1:

```python
PASTA_DADOS = RAIZ / "docs" / "data_tiny"
ID_VEICULO = 1
MAX_ENCOMENDAS = 2
MAX_NOS_EXPANDIDOS = 50_000
```

Para testar o segundo veículo, altera `ID_VEICULO = 2`. Para testar mais encomendas, aumenta `MAX_ENCOMENDAS`, tendo em conta que o espaço de pesquisa pode crescer rapidamente.

Também podes mudar `PASTA_DADOS` para o conjunto `small`. Nesse caso, começa com poucas encomendas e mantém um limite de nós, para evitar uma pesquisa demasiado pesada.

**Importante:** testar o veículo 1 e depois o veículo 2 com o mesmo conjunto de encomendas é útil para testar o módulo, mas não representa uma distribuição real da frota. Não somes essas duas rotas como resultado global, a menos que as encomendas tenham sido atribuídas de forma disjunta pelo CSP.